# Transfer learning: peptide single-task RT → lipid RT (Part B: 75/100%)

Continues `chemberta_TransferLearning_peptide-singletaskRT_to_Lipidmetlin_A.ipynb` in a fresh Colab session.

Upload required artifacts before running:

- `peptide_singletaskRT_bert_encoder.pth`
- `peptide_singletaskRT_rt_scaler.pkl`
- `lipid_metrics_partial_5_25_50_singletaskRT.pkl`
- `METLIN_RT_lipid_train_with_rdkit.csv`
- `METLIN_RT_lipid_test_with_rdkit.csv`

In [ ]:
!pip install torch transformers scikit-learn matplotlib joblib

In [ ]:
n = 1
while n < 2:
    from google.colab import files
    uploaded = files.upload()
    n += 1

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import r2_score, mean_absolute_error

tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

lipid_train_df = pd.read_csv('/content/METLIN_RT_lipid_train_with_rdkit.csv')
lipid_test_df  = pd.read_csv('/content/METLIN_RT_lipid_test_with_rdkit.csv')
print('lipid train:', len(lipid_train_df), '   test:', len(lipid_test_df))

encoder_state = torch.load('peptide_singletaskRT_bert_encoder.pth', map_location=device)
peptide_rt_scaler = joblib.load('peptide_singletaskRT_rt_scaler.pkl')
partial = joblib.load('lipid_metrics_partial_5_25_50_singletaskRT.pkl')
all_rows = list(partial['rows'])
print(f'loaded partial metrics: {len(all_rows)} rows from percentages={partial["percentages_done"]}, seeds={partial["seeds"]}')

In [ ]:
class SmilesRTDataset(Dataset):
    def __init__(self, smiles, rt_scaled, max_length=128):
        self.smiles = smiles
        self.rt = torch.tensor(rt_scaled, dtype=torch.float32)
        self.max_length = max_length
    def __len__(self):
        return len(self.smiles)
    def __getitem__(self, idx):
        enc = tokenizer(self.smiles[idx], padding='max_length', truncation=True,
                        max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'rt':             self.rt[idx],
        }

class ChemBERTaRTRegressor(nn.Module):
    def __init__(self, n_outputs=1):
        super().__init__()
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hs = self.bert.config.hidden_size
        self.regressor = nn.Linear(hs, n_outputs)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.regressor(cls)

def eval_singletask(model, loader, scaler):
    model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for batch in loader:
            p = model(batch['input_ids'].to(device),
                      batch['attention_mask'].to(device)).squeeze(-1).cpu().numpy()
            preds.append(p); ys.append(batch['rt'].numpy())
    p = scaler.inverse_transform(np.concatenate(preds).reshape(-1, 1)).ravel()
    y = scaler.inverse_transform(np.concatenate(ys).reshape(-1, 1)).ravel()
    return r2_score(y, p), mean_absolute_error(y, p)

def train_lipid_model(model, train_loader, test_loader, epochs=15):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5)
    criterion = nn.MSELoss()
    epoch_metrics = []
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            pred = model(batch['input_ids'].to(device),
                         batch['attention_mask'].to(device)).squeeze(-1)
            loss = criterion(pred, batch['rt'].to(device))
            loss.backward()
            optimizer.step()
        r2_tr, mae_tr = eval_singletask(model, train_loader, peptide_rt_scaler)
        r2_te, mae_te = eval_singletask(model, test_loader,  peptide_rt_scaler)
        epoch_metrics.append({'epoch': epoch,
                              'r2_train': r2_tr, 'r2_test': r2_te,
                              'mae_train': mae_tr, 'mae_test': mae_te})
    return epoch_metrics

lipid_rt_train = peptide_rt_scaler.transform(lipid_train_df[['rt']].values).astype(np.float32).ravel()
lipid_rt_test  = peptide_rt_scaler.transform(lipid_test_df[['rt']].values).astype(np.float32).ravel()
lipid_train_smiles = lipid_train_df['smile'].tolist()
lipid_test_smiles  = lipid_test_df['smile'].tolist()
lipid_test_ds      = SmilesRTDataset(lipid_test_smiles, lipid_rt_test)
lipid_test_loader  = DataLoader(lipid_test_ds, batch_size=16, shuffle=False)

PERCENTAGES_PART_B = [75, 100]
SEEDS = partial['seeds']  # match Part A
n_total = len(lipid_train_smiles)

for pct in PERCENTAGES_PART_B:
    n_sub = int(n_total * pct / 100)
    print(f'\n=== Lipid {pct}%  (n={n_sub})  ===')
    for seed in SEEDS:
        rng = np.random.RandomState(42 + seed)
        idx = rng.choice(n_total, size=n_sub, replace=False)
        sub_smiles = [lipid_train_smiles[i] for i in idx]
        sub_rt     = lipid_rt_train[idx]
        sub_ds     = SmilesRTDataset(sub_smiles, sub_rt)
        sub_loader = DataLoader(sub_ds, batch_size=16, shuffle=True)

        torch.manual_seed(seed); np.random.seed(seed)
        for variant_name in ['Transfer', 'Baseline']:
            print(f'  seed={seed}  {variant_name}')
            if variant_name == 'Transfer':
                model = ChemBERTaRTRegressor(n_outputs=1).to(device)
                model.bert.load_state_dict(encoder_state)
            else:
                model = ChemBERTaRTRegressor(n_outputs=1).to(device)
            epoch_metrics = train_lipid_model(model, sub_loader, lipid_test_loader)
            for em in epoch_metrics:
                for split, r2_key, mae_key in [('Train', 'r2_train', 'mae_train'),
                                                ('Test', 'r2_test', 'mae_test')]:
                    all_rows.append({
                        'source_task': 'peptide-singletaskRT',
                        'percentage': pct, 'seed': seed,
                        'model': variant_name, 'epoch': em['epoch'],
                        'split': split,
                        'metric': 'R2',  'value': em[r2_key],
                    })
                    all_rows.append({
                        'source_task': 'peptide-singletaskRT',
                        'percentage': pct, 'seed': seed,
                        'model': variant_name, 'epoch': em['epoch'],
                        'split': split,
                        'metric': 'MAE', 'value': em[mae_key],
                    })
            del model
            torch.cuda.empty_cache() if device.type == 'cuda' else None

final_df = pd.DataFrame(all_rows)
final_df.to_csv('lipid_transfer_singletaskRT_full.csv', index=False)
print(f'\nsaved lipid_transfer_singletaskRT_full.csv ({len(final_df)} rows, {final_df["percentage"].nunique()} fractions)')
summary = (final_df[(final_df['split']=='Test') & (final_df['metric']=='R2')]
           .groupby(['percentage','model'])['value']
           .agg(['median','quantile']).round(4))
print(summary)